# Fan — normal spectrogram statistics for mask sampling

We use **fan** DCASE2020 Task 2 **train** clips (all normal). Each clip is the same representation as `DCASE2020Task2LogMelDataset`: log-mel, **per-file standardized** (`standardize_spectrogram`), padded to `target_T` (320).

Goal: derive **distribution parameters** (not fixed mask values) for four anomaly-map controls:

| Mask parameter | Statistic | Role |
|---|---|---|
| `f0` | Mean energy profile $\mu(b)$ | Categorical weights over mel bins |
| `bandwidth` | Frequency correlation length $\lambda_f$ | Geometric `bw_p` from median adjacent-band correlation length |
| `num_segments` | Temporal energy variance $\sigma^2_t$ | Poisson rate via $\tanh(\sigma^2_t/\text{scale})$ |
| `segment_length` | Temporal autocorrelation $\tau_{\mathrm{half}}$ | Uniform range $[\texttt{lo},\texttt{hi}]$ in frames |

Variety comes from **sampling** each draw; statistics only set the distributions.

## 1. Paths and imports

In [ ]:
import sys
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


def find_project_root() -> Path:
    p = Path.cwd().resolve()
    for _ in range(10):
        if (p / "src" / "data" / "dataset.py").is_file():
            return p
        if p.parent == p:
            break
        p = p.parent
    raise FileNotFoundError("Could not find project root (src/data/dataset.py).")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = Path("/mnt/ssd/LaCie/dcase2020-task2-dev-dataset")
if not DATA_PATH.exists():
    DATA_PATH = Path("/mnt/ssd/LaCie/dcase2020_task2/dcase2020_task2_dev_dataset")
if not DATA_PATH.exists():
    DATA_PATH = PROJECT_ROOT / "data" / "dcase2020-task2-dev-dataset"
if not DATA_PATH.exists():
    DATA_PATH = PROJECT_ROOT / "dataset" / "dcase2020_task2_dev_dataset"
if not DATA_PATH.exists():
    DATA_PATH = Path("../../data/dcase2020-task2-dev-dataset").resolve()

MACHINE_TYPE = "fan"
RNG = np.random.default_rng(0)

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DATA_PATH: {DATA_PATH} (exists={DATA_PATH.exists()})")

## 2. Load fan normal training spectrograms `(N, n_mels, T)`

In [ ]:
from src.data.dataset import DCASE2020Task2LogMelDataset

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Missing dataset root: {DATA_PATH}")

train_ds = DCASE2020Task2LogMelDataset(root=str(DATA_PATH), machine_type=MACHINE_TYPE)
# Stored as (N, 1, n_mels, T) — same tensors as __getitem__ without RGB conversion
raw = train_ds.data.numpy()
normal_specs = raw[:, 0, :, :].astype(np.float64)
N, n_mels, T = normal_specs.shape
print(f"normal_specs: shape={normal_specs.shape}, dtype={normal_specs.dtype}")

## 3. `MaskSamplingParams`, `compute_mask_sampling_params`, `sample_mask_params`

Implements the mapping you described. `sample_mask_params` uses a `numpy.random.Generator` (`integers` for segment length).

In [ ]:
@dataclass
class MaskSamplingParams:
    """
    Sampling distribution parameters derived from normal spectrogram statistics.
    One instance per machine_type.
    """

    band_weights: np.ndarray  # (n_mels,) sums to 1
    bw_p: float  # geometric success probability
    seg_lambda: float  # Poisson rate (clipped at sample time)
    seg_len_lo: int
    seg_len_hi: int


def _acf_mean_trace(band_series: np.ndarray, max_lag: int, corr_threshold: float) -> np.ndarray:
    """Mean log-mel trace (over files) vs lag; r=1 at lag 0."""
    s = band_series.mean(axis=0)
    out = np.ones(max_lag, dtype=np.float64)
    for lag in range(1, max_lag):
        a, b = s[:-lag], s[lag:]
        if a.size < 2:
            out[lag:] = 0.0
            break
        r = np.corrcoef(a, b)[0, 1]
        if np.isnan(r):
            r = 0.0
        out[lag] = r
    return out


def _mask_sampling_with_diag(
    normal_specs: np.ndarray,
    temperature: float = 0.5,
    uniform_mix: float = 0.3,
    corr_threshold: float = 0.5,
    min_seg_len_frac: float = 0.05,
    max_seg_len_frac: float = 0.40,
) -> tuple[MaskSamplingParams, dict]:
    N, n_mels, T = normal_specs.shape

    # --- 1. f0: mean energy profile mu(b) ---
    mu = normal_specs.mean(axis=(0, 2))
    mu_shifted = mu - mu.min()
    softmax_w = np.exp(mu_shifted / temperature)
    softmax_w /= softmax_w.sum()
    uniform_w = np.ones(n_mels) / n_mels
    band_weights = (1 - uniform_mix) * softmax_w + uniform_mix * uniform_w

    # --- 2. bandwidth: frequency correlation length lambda_f ---
    flat = normal_specs.reshape(-1, n_mels)
    C = np.corrcoef(flat.T)
    C = np.nan_to_num(C, nan=0.0, posinf=1.0, neginf=-1.0)

    corr_lengths = []
    for b in range(n_mels):
        length = 1
        for k in range(1, n_mels - b):
            if C[b, b + k] >= corr_threshold:
                length += 1
            else:
                break
        corr_lengths.append(length)

    lambda_f = int(np.median(corr_lengths))
    lambda_f = max(1, lambda_f)

    bw_p = 1.0 / (2.0 * lambda_f)
    bw_p = float(np.clip(bw_p, 0.05, 0.9))

    # --- 3. num_segments: temporal variance sigma2_t ---
    temporal_var = normal_specs.var(axis=2)
    sigma2_t = float(temporal_var.mean())

    scale = 0.5
    max_lambda = 4.0
    seg_lambda = 1.0 + (max_lambda - 1.0) * np.tanh(sigma2_t / scale)

    # --- 4. segment_length: tau_half from temporal ACF ---
    max_lag = max(1, T // 2)
    tau_halfs = []
    for b in range(n_mels):
        band_series = normal_specs[:, b, :]
        acf = _acf_mean_trace(band_series, max_lag, corr_threshold)
        tau = next((lag for lag, r in enumerate(acf) if r < corr_threshold), max_lag)
        tau_halfs.append(tau)

    tau_half = int(np.median(tau_halfs))
    tau_half = max(1, tau_half)

    seg_len_lo = int(np.clip(tau_half, min_seg_len_frac * T, max_seg_len_frac * T))
    seg_len_hi = int(np.clip(4 * tau_half, seg_len_lo + 1, max_seg_len_frac * T))

    return MaskSamplingParams(
        band_weights=band_weights.astype(np.float32),
        bw_p=bw_p,
        seg_lambda=float(seg_lambda),
        seg_len_lo=seg_len_lo,
        seg_len_hi=seg_len_hi,
    ), {
        "mu": mu,
        "C": C,
        "corr_lengths": np.array(corr_lengths, dtype=np.int32),
        "lambda_f": lambda_f,
        "sigma2_t": sigma2_t,
        "temporal_var": temporal_var,
        "tau_halfs": np.array(tau_halfs, dtype=np.int32),
        "tau_half": tau_half,
    }


def compute_mask_sampling_params(
    normal_specs: np.ndarray,
    temperature: float = 0.5,
    uniform_mix: float = 0.3,
    corr_threshold: float = 0.5,
    min_seg_len_frac: float = 0.05,
    max_seg_len_frac: float = 0.40,
) -> MaskSamplingParams:
    return _mask_sampling_with_diag(
        normal_specs,
        temperature,
        uniform_mix,
        corr_threshold,
        min_seg_len_frac,
        max_seg_len_frac,
    )[0]


def sample_mask_params(
    p: MaskSamplingParams,
    n_mels: int,
    T: int,
    max_segments: int = 8,
    rng: np.random.Generator | None = None,
) -> tuple[int, int, int, int]:
    rng = rng or np.random.default_rng()
    f0 = int(rng.choice(n_mels, p=p.band_weights))

    bw = int(rng.geometric(p.bw_p))
    min_bw = max(1, round(1.0 / (2.0 * p.bw_p)))
    bw = int(np.clip(bw + min_bw, 1, n_mels - f0))

    n_seg = int(np.clip(rng.poisson(p.seg_lambda), 1, max_segments))

    lo, hi = p.seg_len_lo, p.seg_len_hi
    if hi < lo:
        lo, hi = hi, lo
    seg_len = int(rng.integers(lo, hi + 1))

    return f0, bw, n_seg, seg_len

## 4. Fan: fitted sampling parameters + intermediate statistics

In [ ]:
p_fan, diag = _mask_sampling_with_diag(normal_specs)

print("MaskSamplingParams (fan, normal train):")
print(f"  band_weights: sum={p_fan.band_weights.sum():.6f}, min={p_fan.band_weights.min():.5e}, max={p_fan.band_weights.max():.5f}")
print(f"  bw_p (Geometric)     = {p_fan.bw_p:.4f}  (mean of raw Geom ≈ {1/p_fan.bw_p:.2f})")
print(f"  seg_lambda (Poisson) = {p_fan.seg_lambda:.4f}")
print(f"  seg_len_lo, seg_len_hi = {p_fan.seg_len_lo}, {p_fan.seg_len_hi}  (T={T})")
print()
print("Intermediate:")
print(f"  lambda_f (median corr length) = {diag['lambda_f']}")
print(f"  sigma2_t (mean temporal var)  = {diag['sigma2_t']:.6f}")
print(f"  tau_half (median ACF crossing) = {diag['tau_half']}")

### 4a. $\mu(b)$ and band weights (f0)

Softmax energy emphasis + `uniform_mix` keeps every bin reachable.

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
bins = np.arange(n_mels)
ax[0].plot(bins, diag["mu"], color="C0")
ax[0].set_ylabel(r"$\mu(b)$ mean log-mel (stdized)")
ax[0].set_title("Fan — mean profile and f0 categorical weights")
ax[1].bar(bins, p_fan.band_weights, width=1.0, color="C1", alpha=0.85)
ax[1].set_ylabel("band_weights")
ax[1].set_xlabel("mel bin")
plt.tight_layout()
plt.show()

### 4b. Band–band correlation and $\lambda_f$

Left: subset of $C$. Right: per-start-bin correlation length (walk right until $r < 0.5$).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sub = 32
im = ax[0].imshow(diag["C"][:sub, :sub], cmap="coolwarm", vmin=-1, vmax=1, aspect="auto")
ax[0].set_title(f"C[:{sub},:{sub}] (Pearson between mel bins)")
plt.colorbar(im, ax=ax[0], fraction=0.046)
ax[1].plot(diag["corr_lengths"], marker=".", linestyle="none")
ax[1].axhline(diag["lambda_f"], color="C3", label=f"median = λ_f = {diag['lambda_f']}")
ax[1].set_xlabel("start band b")
ax[1].set_ylabel("corr length")
ax[1].legend()
ax[1].set_title("Adjacent correlation run length")
plt.tight_layout()
plt.show()

### 4c. $\sigma^2_t(b)$ per band and global $\sigma^2_t$

`temporal_var`: variance over time per file and band; scalar `sigma2_t` is the mean over files and bands.

In [ ]:
tv_mean = diag["temporal_var"].mean(axis=0)
fig, ax = plt.subplots(1, 1, figsize=(10, 3))
ax.plot(tv_mean, label=r"$\mathbb{E}_n[\mathrm{Var}_t x_{n,b,t}]$")
ax.axhline(diag["sigma2_t"], color="C3", linestyle="--", label=f"sigma2_t mean = {diag['sigma2_t']:.4f}")
ax.set_xlabel("mel bin")
ax.set_ylabel("temporal variance")
ax.legend()
ax.set_title("Per-band temporal energy variance (fan normal)")
plt.tight_layout()
plt.show()

### 4d. Example ACF (mean trace) and $\tau_{\mathrm{half}}$ distribution across bands

In [ ]:
max_lag = max(1, T // 2)
show_bands = [0, n_mels // 4, n_mels // 2, 3 * n_mels // 4, n_mels - 1]
fig, ax = plt.subplots(1, 1, figsize=(9, 4))
for b in show_bands:
    acf = _acf_mean_trace(normal_specs[:, b, :], max_lag, 0.5)
    ax.plot(acf, label=f"b={b}")
ax.axhline(0.5, color="k", linestyle=":", alpha=0.5)
ax.set_xlabel("lag (frames)")
ax.set_ylabel("ACF")
ax.set_title("Mean-trace temporal ACF (example bands)")
ax.legend(ncol=3, fontsize=8)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(1, 1, figsize=(8, 3))
ax.hist(diag["tau_halfs"], bins=min(40, n_mels), color="C2", edgecolor="white")
ax.axvline(diag["tau_half"], color="C3", label=f"median tau_half = {diag['tau_half']}")
ax.set_xlabel(r"$\tau_{\mathrm{half}}$ (first lag with ACF < 0.5)")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Monte Carlo: one distribution per knob

Many independent draws from `sample_mask_params` using **fan** `MaskSamplingParams`. This is what training-time randomisation would look like.

In [ ]:
n_draw = 20_000
rows = [sample_mask_params(p_fan, n_mels, T, rng=RNG) for _ in range(n_draw)]
f0s, bws, n_segs, seg_lens = zip(*rows)
f0s = np.array(f0s)
bws = np.array(bws)
n_segs = np.array(n_segs)
seg_lens = np.array(seg_lens)

fig, axes = plt.subplots(2, 2, figsize=(10, 7))
axes[0, 0].hist(f0s, bins=np.arange(n_mels + 1) - 0.5, color="C0", edgecolor="white")
axes[0, 0].set_title("f0 ~ Categorical(band_weights)")
axes[0, 0].set_xlabel("mel bin")

axes[0, 1].hist(bws, bins=min(50, bws.max()), color="C1", edgecolor="white")
axes[0, 1].set_title("bandwidth ~ Geometric(bw_p) + min_bw")
axes[0, 1].set_xlabel("bins wide")

axes[1, 0].hist(n_segs, bins=np.arange(1, n_segs.max() + 2) - 0.5, color="C2", edgecolor="white")
axes[1, 0].set_title("num_segments ~ Poisson(seg_lambda) clipped")
axes[1, 0].set_xlabel("count")

axes[1, 1].hist(seg_lens, bins="auto", color="C3", edgecolor="white")
axes[1, 1].set_title("segment_length ~ Uniform(lo, hi)")
axes[1, 1].set_xlabel("frames")

plt.tight_layout()
plt.show()

print("Sample summary (Monte Carlo):")
print(f"  f0:         mean={f0s.mean():.1f}, std={f0s.std():.1f}")
print(f"  bandwidth:  mean={bws.mean():.2f}, std={bws.std():.2f}")
print(f"  n_seg:      mean={n_segs.mean():.2f}, P(n=1)={(n_segs==1).mean():.3f}")
print(f"  seg_len:    mean={seg_lens.mean():.2f}, min={seg_lens.min()}, max={seg_lens.max()}")

## 6. Notes — loosening toward a flat prior

- **f0**: increase `uniform_mix` → closer to uniform over bins.
- **bandwidth**: `bw_p` is derived from $\lambda_f$; scale the `1/(2λ_f)` rule or clip range to bias wide vs tight.
- **num_segments**: lower the `tanh` `scale` (more sensitive to $\sigma^2_t$) or change `max_lambda`.
- **segment_length**: `min_seg_len_frac` / `max_seg_len_frac` clip the uniform support in units of $T$.

Next step (outside this notebook): thread `MaskSamplingParams` per `machine_type` into `AnomalyMapGenerator` / `anomaly_map.py` so each draw calls `sample_mask_params` instead of fixed hyperparameters.